# Hermes + Google Colab

## Getting Colab MCP Running

#### **Step 1: Open a Colab notebook**

- Go to https://colab.research.google.com,
- Create a new notebook
- Change runtime to T4 GPU (Runtime → Change runtime type → T4 GPU).

---

#### **Step 2: Connect the MCP**

The Colab MCP server is now in your Claude Code settings. Next time you start a Claude Code session, it will register. You'll be able to ask Claude Code to execute code directly on Colab's T4 GPU.

The MCP works via your browser session — you need the Colab notebook tab open in your browser.

---

#### **Step 3: For Hermes (persistent inference)**

If you want Hermes in k3d to use Colab's GPU, run the cells below

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Go to Runtime > Change runtime type > T4 GPU > Save.")

gpu_name = torch.cuda.get_device_name(0)
print("Current GPU:", gpu_name)

if "T4" not in gpu_name:
    raise RuntimeError("This is not a T4 GPU. Change runtime type to T4 GPU manually.")

print("✅ T4 GPU is enabled")

Current GPU: Tesla T4
✅ T4 GPU is enabled


In [2]:
  # Cell 1: Install & start Ollama
  !curl -fsSL https://ollama.ai/install.sh | sh
  import subprocess, time
  subprocess.Popen(["ollama", "serve"])
  time.sleep(5)

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [ ]:
  # Cell 2: Pull a real model (8B fits in T4's 16GB VRAM)
  !ollama pull hermes3:8b

In [ ]:
  # Cell 3: Expose via cloudflared tunnel
  !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  !chmod +x cloudflared-linux-amd64
  import subprocess
  proc = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:11434"],
                           stderr=subprocess.PIPE, text=True)
  for line in proc.stderr:
      if "trycloudflare.com" in line:
print(f"\n>>> TUNNEL URL: {line.strip()}")
break